# Stage 6.2 -- Focused T-side A/B Feature Refinement Experiment

Controlled comparison of feature sets intended to reduce B-predicted-as-A errors.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data/gold/modeling/t_side_ab_refined_experiment'

def load(name):
    return pd.read_parquet(DATA / f'{name}.parquet')

audit = load('ab_refined_dataset_audit')
feature_sets = load('ab_refined_feature_sets')
metrics = load('ab_refined_metrics')
predictions = load('ab_refined_predictions')
importance = load('ab_refined_feature_importance')
errors = load('ab_refined_error_summary')
comparison = load('ab_refined_comparison_vs_baseline')
recommendation = load('ab_refined_recommendation')

## Dataset and feature sets

In [ ]:
display(audit.T.rename(columns={0: 'value'}))
display(feature_sets[['horizon_seconds', 'feature_set_name', 'model_rows', 'total_selected_features', 'notes']])

## Metrics and Stage 6 comparison

In [ ]:
display(metrics[['horizon_seconds', 'feature_set_name', 'model_name', 'macro_f1', 'balanced_accuracy', 'recall_B', 'B_predicted_as_A']])
display(comparison)
plot_data = comparison[comparison['model_name'].eq('logistic_regression')]
for name, group in plot_data.groupby('feature_set_name'):
    plt.plot(group['horizon_seconds'], group['delta_recall_B'], marker='o', label=name)
plt.axhline(0, color='black', linewidth=1)
plt.title('Recall B improvement vs Stage 6')
plt.xlabel('Horizon seconds')
plt.ylabel('Delta recall B')
plt.legend()
plt.show()

## B-predicted-as-A errors

In [ ]:
display(errors.sort_values(['B_predicted_as_A', 'high_confidence_B_predicted_as_A']).head(30))

## Feature importance

In [ ]:
top_importance = (importance.assign(abs_importance=importance['importance_value'].abs())
                  .sort_values(['horizon_seconds', 'feature_set_name', 'model_name', 'abs_importance'], ascending=[True, True, True, False])
                  .groupby(['horizon_seconds', 'feature_set_name', 'model_name']).head(5))
display(top_importance[['horizon_seconds', 'feature_set_name', 'model_name', 'feature_name', 'importance_value', 'direction']])

## Final recommendation and best-candidate errors

In [ ]:
display(recommendation.head(15))
best = recommendation.iloc[0]
best_errors = predictions[(predictions['horizon_seconds'].eq(best['horizon_seconds'])) &
                          (predictions['feature_set_name'].eq(best['feature_set_name'])) &
                          (predictions['model_name'].eq(best['model_name'])) &
                          (~predictions['is_correct'])]
display(best_errors.sort_values('prediction_confidence', ascending=False).head(20))

## Next

Next: choose candidate baseline or perform manual review before final modeling